In [13]:
import pandas as pd

df = pd.read_excel("./../csv/library_id_cond.xlsx")

In [14]:
df['library_id'] = df['library_id'].str.replace("_","-")

In [15]:
df = df[df['cond']!= "UNC"]

In [16]:
df

,library_id,cond
0,18-57617-A1,IPF
1,20-33940-B2,IPF
2,20-24241-A2,IPF
3,19-35057-C3,NSIP
4,20-17688-B2,NSIP
5,20-28197-A1,IPF
6,20-22642-A1,NSIP
7,20-41501-C1,IPF
8,20-33362-C4,NSIP
10,20-41615-B1,IPF


In [17]:
df['IPF_onehot']  = (df['cond']=='IPF').astype(int)

In [18]:
df

,library_id,cond,IPF_onehot
0,18-57617-A1,IPF,1
1,20-33940-B2,IPF,1
2,20-24241-A2,IPF,1
3,19-35057-C3,NSIP,0
4,20-17688-B2,NSIP,0
5,20-28197-A1,IPF,1
6,20-22642-A1,NSIP,0
7,20-41501-C1,IPF,1
8,20-33362-C4,NSIP,0
10,20-41615-B1,IPF,1


In [19]:
df = df.reset_index(drop = True)

In [20]:
df.shape

(29, 3)

In [21]:
from sklearn.model_selection import LeaveOneOut

# df is your existing DataFrame with 29 rows

loo = LeaveOneOut()

# Create a column for each fold
for fold_idx, (train_idx, test_idx) in enumerate(loo.split(df)):
    colname = f"fold_{fold_idx}"
    
    df.loc[:, colname] = "train"   # default for all rows
    df.loc[test_idx, colname] = "test"   # mark the one test sample

In [22]:
df

,library_id,cond,IPF_onehot,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,...,fold_19,fold_20,fold_21,fold_22,fold_23,fold_24,fold_25,fold_26,fold_27,fold_28
0,18-57617-A1,IPF,1,test,train,train,train,train,train,train,...,train,train,train,train,train,train,train,train,train,train
1,20-33940-B2,IPF,1,train,test,train,train,train,train,train,...,train,train,train,train,train,train,train,train,train,train
2,20-24241-A2,IPF,1,train,train,test,train,train,train,train,...,train,train,train,train,train,train,train,train,train,train
3,19-35057-C3,NSIP,0,train,train,train,test,train,train,train,...,train,train,train,train,train,train,train,train,train,train
4,20-17688-B2,NSIP,0,train,train,train,train,test,train,train,...,train,train,train,train,train,train,train,train,train,train
5,20-28197-A1,IPF,1,train,train,train,train,train,test,train,...,train,train,train,train,train,train,train,train,train,train
6,20-22642-A1,NSIP,0,train,train,train,train,train,train,test,...,train,train,train,train,train,train,train,train,train,train
7,20-41501-C1,IPF,1,train,train,train,train,train,train,train,...,train,train,train,train,train,train,train,train,train,train
8,20-33362-C4,NSIP,0,train,train,train,train,train,train,train,...,train,train,train,train,train,train,train,train,train,train
9,20-41615-B1,IPF,1,train,train,train,train,train,train,train,...,train,train,train,train,train,train,train,train,train,train


In [23]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import h5py
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score

from trident.slide_encoder_models import ABMILSlideEncoder

# Set deterministic behavior
SEED = 1234
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


class BinaryClassificationModel(nn.Module):
    def __init__(self, input_feature_dim=1536, n_heads=1, head_dim=512, dropout=0., gated=True, hidden_dim=256):
        super().__init__()
        self.feature_encoder = ABMILSlideEncoder(
            freeze=False,
            input_feature_dim=input_feature_dim, 
            n_heads=n_heads, 
            head_dim=head_dim, 
            dropout=dropout, 
            gated=gated
        )
        self.classifier = nn.Sequential(
            nn.Linear(input_feature_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, return_raw_attention=False):
        if return_raw_attention:
            features, attn = self.feature_encoder(x, return_raw_attention=True)
        else:
            features = self.feature_encoder(x)
        logits = self.classifier(features).squeeze(1)
        
        if return_raw_attention:
            return logits, attn
        
        return logits

# Initialize model
device = 'mps'
model = BinaryClassificationModel().to(device)

# Custom dataset
class H5Dataset(Dataset):
    def __init__(self, feats_path, df, split, test_idx = None, num_features=1536):
        #self.df = df[df["fold_0"] == split]
        #self.df = df.drop(test_idx)
        self.df = df[df[f"fold_{test_idx}"] == split]
        self.feats_path = feats_path
        self.num_features = num_features
        self.split = split
    
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        with h5py.File(os.path.join(self.feats_path, row['library_id'] + '.h5'), "r") as f:
            features = torch.from_numpy(f["features"][:])

        if self.split == 'train':
            num_available = features.shape[0]
            if num_available >= self.num_features:
                indices = torch.randperm(num_available, generator=torch.Generator().manual_seed(SEED))[:self.num_features]
            else:
                indices = torch.randint(num_available, (self.num_features,), generator=torch.Generator().manual_seed(SEED))  # Oversampling
            features = features[indices]

        label = torch.tensor(row["IPF_onehot"], dtype=torch.float32)
        return features, label

# # Create dataloaders
# feats_path = '/Users/jk/Documents/Lab2/visium/python_analysis/trident/svs/trident_processed/20x_256px_0px_overlap/features_uni_v2'  #'./tutorial-3/cptac_ccrcc/20x_512px_0px_overlap/features_conch_v15'
# batch_size = 22
# train_loader = DataLoader(H5Dataset(feats_path, df, "train"), batch_size=batch_size, shuffle=False, worker_init_fn=lambda _: np.random.seed(SEED))
# test_loader = DataLoader(H5Dataset(feats_path, df, "test"), batch_size=1, shuffle=False, worker_init_fn=lambda _: np.random.seed(SEED))





In [24]:
# Stuff that doesnt change 
feats_path = '/Users/jk/Documents/Lab2/visium/python_analysis/cpath/trident/svs/trident_processed/20x_256px_0px_overlap/features_uni_v2'  #'./tutorial-3/cptac_ccrcc/20x_512px_0px_overlap/features_conch_v15'
batch_size = 28
# Training setup
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=4e-4)

all_labels = [] # to be a list of array where each index contains labels from one CV fold
all_outputs = []
# Put ABMIL inside LOOCV loop
for i in range(len(df)):
    print(f"LOOCV fold {i+1}/{len(df)}")

    # Split
    train_dataset = H5Dataset(feats_path, df, "train", test_idx=i)
    test_dataset = H5Dataset(feats_path, df, "test", test_idx=i)  # single sample
    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, worker_init_fn=lambda _: np.random.seed(SEED))
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

    # Initialize fresh model per fold
    model = BinaryClassificationModel().to(device)
    optimizer = optim.Adam(model.parameters(), lr=4e-4)
    criterion = nn.BCEWithLogitsLoss()

    # Train model
    num_epochs = 20
    for epoch in range(num_epochs):
        model.train()
        for features, labels in train_loader:
            features, labels = {'features': features.to(device)}, labels.to(device)
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

    # Evaluate on held-out sample
    model.eval()
    with torch.no_grad():
        for features, labels in test_loader:
            features, labels = {'features': features.to(device)}, labels.to(device)
            outputs = model(features)
            all_outputs.append(outputs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())


LOOCV fold 1/29
LOOCV fold 2/29
LOOCV fold 3/29
LOOCV fold 4/29
LOOCV fold 5/29
LOOCV fold 6/29
LOOCV fold 7/29
LOOCV fold 8/29
LOOCV fold 9/29
LOOCV fold 10/29
LOOCV fold 11/29
LOOCV fold 12/29
LOOCV fold 13/29
LOOCV fold 14/29
LOOCV fold 15/29
LOOCV fold 16/29
LOOCV fold 17/29
LOOCV fold 18/29
LOOCV fold 19/29
LOOCV fold 20/29
LOOCV fold 21/29
LOOCV fold 22/29
LOOCV fold 23/29
LOOCV fold 24/29
LOOCV fold 25/29
LOOCV fold 26/29
LOOCV fold 27/29
LOOCV fold 28/29
LOOCV fold 29/29


In [25]:
from sklearn.metrics import roc_auc_score, confusion_matrix
import numpy as np

all_labels = np.concatenate(all_labels)
all_outputs = np.concatenate(all_outputs)

# AUC
auc = roc_auc_score(all_labels, all_outputs)

# Binary predictions
preds = (all_outputs > 0).astype(int)  # logit > 0

# Accuracy
accuracy = (preds == all_labels).mean()

# Confusion matrix
cm = confusion_matrix(all_labels, preds)

print(f"LOOCV Accuracy: {accuracy:.4f}")
print(f"LOOCV AUC: {auc:.4f}")
print("Confusion Matrix:")
print(cm)

LOOCV Accuracy: 0.4828
LOOCV AUC: 0.5263
Confusion Matrix:
[[12  7]
 [ 8  2]]


In [26]:
np.savez("epoch_20_loocv_results", all_labels = all_labels, all_outputs = all_outputs)

In [27]:
all_labels

array([1., 1., 1., 0., 0., 1., 0., 1., 0., 1., 1., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1., 0.], dtype=float32)

In [28]:
all_outputs

array([ -8.482809  ,  -0.02368124,  -0.31741467,  -9.799468  ,
         3.9377115 ,  -1.80482   ,   2.828405  ,   0.3454166 ,
         1.3944812 ,  -2.5673292 ,  -1.7646949 ,  -5.0563493 ,
       -14.459927  , -19.325603  ,   3.803057  , -12.491886  ,
         0.05081925, -14.087358  ,  -5.3941603 ,   1.604019  ,
       -10.093404  , -11.074264  ,  -7.8587017 ,  -1.4311082 ,
        -3.263706  ,  -5.2225614 ,   0.9627194 , -18.63922   ,
         1.3763747 ], dtype=float32)

In [29]:
preds

array([0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0,
       0, 0, 0, 0, 1, 0, 1])